Install the python package pymerkle, if you haven't yet (requires python ver 3.10+)

In [1]:
pip install pymerkle

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Prágai Bálint\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Reading the sinthetic user data

In [ ]:
import pandas as pd

df = pd.read_csv('./generated_table_data/users.csv')

df.iloc[0]

id                                       1
user_full_name                      User 1
city                                London
gender                                male
age                                     66
email                    user1@example.com
phone                          35840958516
bookingtime            2025-01-01 09:00:00
complaint                Burning sensation
satisfaction_rating                      4
allergy                             Pollen
Name: 0, dtype: object

In [3]:
"""
Helper functions for pseudo-anonymization of the data.
"""

def age_bucketer(age):
    match age:
        case age if age < 18:
            return 'under 18'
        case age if age < 25:
            return '18-25'
        case age if age < 40:
            return '25-39'
        case age if age < 66:
            return '40-65'
        case age if age < 80:
            return '66-79'
        case _:
            return '80+'
        
def geopolygon_bucketer(geopolygon):
    match geopolygon:
        case geopolygon if geopolygon == "Helsinki":
            return 'FI-Uusimaa'
        case geopolygon if geopolygon == "Tampere":
                return 'FI-Pirkanmaa'
        case geopolygon if geopolygon == "Stockholm":
            return 'SE-Stockholm'
        case geopolygon if geopolygon == "Berlin":
            return 'DE-Berlin'
        case geopolygon if geopolygon == "London":
            return 'UK-London'
        case geopolygon if geopolygon == "Austin":
            return 'US-Austin'
        case geopolygon if geopolygon == "Tokyo":
            return 'JP-Tokyo'
        case geopolygon if geopolygon == "Singapore":
            return 'SG-Singapore'
        case _:
            return 'other'

### Fixmetree ###
Below is the FixmeTree, a class based on the BaseMerkleTree, customized for this application.

In [4]:
from pymerkle import BaseMerkleTree

class FixmeTree(BaseMerkleTree):

    def __init__(self, algorithm='sha256'):
        """
        Storage setup and superclass initialization
        """
        self.hashes = []

        super().__init__(algorithm)

    @staticmethod
    def preprocess_entry(data):
        """
        Preprocesses data entry before encoding
        e.g. {age: 25, city: "Helsinki"} -> b'{"age_range": 18-25, "geopolygon": "FI-Uusimaa"}'
        """
        if isinstance(data, pd.Series):
            data = data.copy()
            data['age_range'] = age_bucketer(data['age'])
            data['geopolygon'] = geopolygon_bucketer(data['city'])
            data = data.drop(labels=['user_full_name', 'email', 'phone', 'age', 'city'])
            
        return data
    
    def _encode_entry(self, data):
        """
        Prepares data entry for hashing
        """
        proc_data = str(data)
        encoded_data = proc_data.encode('utf-8')
        return encoded_data
        
    def _store_leaf(self, data, digest):
        """
        Stores data hash in a new leaf and returns index
        """
        self.hashes += [digest]

        return len(self.hashes)


    def _get_leaf(self, index):
        """
        Returns the hash stored by the leaf specified
        """
        value = self.hashes[index - 1]

        return value


    def _get_leaves(self, offset, width):
        """
        Returns hashes corresponding to the specified leaf range
        """
        values = self.hashes[offset: offset + width]

        return values


    def _get_size(self):
        """
        Returns the current number of leaves
        """
        return len(self.hashes)

### Small scale demonstration ###
tree_medium is encoding a single row (= a single user) into a Merkle tree.
Per user encoding allows for per-row criteria matching.

In [5]:
tree_medium = FixmeTree(algorithm='sha256')
data = FixmeTree.preprocess_entry(df.iloc[0])
print(data)
for i in range(len(data)):
    tree_medium.append_entry(data.iloc[i])

print(f'tree size: {tree_medium.get_size()}')
print(f'tree state: {tree_medium.get_state().hex()}')

id                                       1
gender                                male
bookingtime            2025-01-01 09:00:00
complaint                Burning sensation
satisfaction_rating                      4
allergy                             Pollen
age_range                            66-79
geopolygon                       UK-London
Name: 0, dtype: object
tree size: 8
tree state: 5908decf242bb29d443cb6ccfec1c6a65ae867af160915f274c0f43b7bc7bd2f


### Small scale proof generation and verification ###
Due to some limitations of the current version of the package, the best way to prove data is within the entry is to create a single leaf tree.
As can be seen in the first example below, verify inclusion does not give and error, meaning the first entry's hash is indeed equal to the hash of the single leaf tree, both encoding the id '1'.
The second example shows, that id '2' is not included.

In [6]:
from pymerkle import verify_inclusion

prove_tree_true = FixmeTree(algorithm='sha256')
prove_tree_true.append_entry('male')

proof = tree_medium.prove_inclusion(2,8)
base = prove_tree_true.get_leaf(1)
root = tree_medium.get_state(8)

verify_inclusion(base, root, proof)

In [8]:
prove_tree_forged = FixmeTree(algorithm='sha256')
prove_tree_forged.append_entry('female')

proof = tree_medium.prove_inclusion(2,8)
base = prove_tree_forged.get_leaf(1)
root = tree_medium.get_state(8)

verify_inclusion(base, root, proof)

InvalidProof: Base hash does not match

### Medium scale example ###
In this example we can see how the algorith performs on 1000 users. All of them are being attribute-wise encoded and the appended to the Merkle-tree.

In [9]:
tree_large = FixmeTree(algorithm='sha256')

for i in range(1000):
    data = FixmeTree.preprocess_entry(df.iloc[i])
    for j in range(len(data)):
        print(f'entry {i} leaf {j}')
        tree_large.append_entry(data[j])

print(f'tree size: {tree_large.get_size()}')
print(f'tree state: {tree_large.get_state().hex()}')

entry 0 leaf 0
entry 0 leaf 1
entry 0 leaf 2
entry 0 leaf 3
entry 0 leaf 4
entry 0 leaf 5
entry 0 leaf 6
entry 0 leaf 7
entry 1 leaf 0
entry 1 leaf 1
entry 1 leaf 2
entry 1 leaf 3
entry 1 leaf 4
entry 1 leaf 5
entry 1 leaf 6
entry 1 leaf 7
entry 2 leaf 0
entry 2 leaf 1
entry 2 leaf 2
entry 2 leaf 3
entry 2 leaf 4
entry 2 leaf 5
entry 2 leaf 6
entry 2 leaf 7
entry 3 leaf 0
entry 3 leaf 1
entry 3 leaf 2
entry 3 leaf 3
entry 3 leaf 4
entry 3 leaf 5
entry 3 leaf 6
entry 3 leaf 7
entry 4 leaf 0
entry 4 leaf 1
entry 4 leaf 2
entry 4 leaf 3
entry 4 leaf 4
entry 4 leaf 5
entry 4 leaf 6
entry 4 leaf 7
entry 5 leaf 0
entry 5 leaf 1
entry 5 leaf 2
entry 5 leaf 3
entry 5 leaf 4
entry 5 leaf 5
entry 5 leaf 6
entry 5 leaf 7
entry 6 leaf 0
entry 6 leaf 1
entry 6 leaf 2
entry 6 leaf 3
entry 6 leaf 4
entry 6 leaf 5
entry 6 leaf 6
entry 6 leaf 7
entry 7 leaf 0
entry 7 leaf 1
entry 7 leaf 2
entry 7 leaf 3
entry 7 leaf 4
entry 7 leaf 5
entry 7 leaf 6
entry 7 leaf 7
entry 8 leaf 0
entry 8 leaf 1
entry 8 le

## Large scale encoding experiment ##
This time, we are creating a Merkle-tree for each user in the whole example database, meaning 10 000 Merkle trees; all with 8 leaves.

In [ ]:
encoded_df = []

for i in range(len(df)):
    entry = None
    locals()["user_tree_" + str(i)] = FixmeTree(algorithm='sha256')
    data = FixmeTree.preprocess_entry(df.iloc[i])
    for j in range(len(data)):
        #print(f'entry {i} leaf {j}')
        locals()["user_tree_" + str(i)].append_entry(data[j])
    encoded_df.append(locals()["user_tree_" + str(i)])

hex_data = pd.DataFrame(encoded_df)
hex_data

,0
0,<__main__.FixmeTree object at 0x000002416BAE78E0>
1,<__main__.FixmeTree object at 0x000002416B7277F0>
2,<__main__.FixmeTree object at 0x000002416B5E40A0>
3,<__main__.FixmeTree object at 0x000002416BAE73D0>
4,<__main__.FixmeTree object at 0x000002416BAE6650>
...,...
9995,<__main__.FixmeTree object at 0x000002416DB8BC70>
9996,<__main__.FixmeTree object at 0x000002416DBD0640>
9997,<__main__.FixmeTree object at 0x000002416DBD02B0>
9998,<__main__.FixmeTree object at 0x000002416DBD02E0>


### proof generation ###
Now we check if our encodings actually can prove inclusion:

In [ ]:
check = FixmeTree.preprocess_entry(df.iloc[0])
print(check)
check['age_range']

id                                       1
gender                                male
bookingtime            2025-01-01 09:00:00
complaint                Burning sensation
satisfaction_rating                      4
allergy                             Pollen
age_range                            66-79
geopolygon                       UK-London
Name: 0, dtype: object


'66-79'

In [ ]:
from pymerkle import verify_inclusion

user_tree = hex_data.iloc[0][0] #This is a FixMeTree object, not a hash
query_tree = FixmeTree(algorithm='sha256')
query_tree.append_entry(check['age_range'])

proof = user_tree.prove_inclusion(7, 8)
base = query_tree.get_leaf(1)
root = user_tree.get_state(8)

verify_inclusion(base, root, proof)


#### Hurray ####
And we get no errors, which means the proof is valid!
Now let's see an application for this on the whole dataset; How many of the participants are from the "UK-London" area?

In [ ]:
def test_geolocation_match(dataframe:pd.DataFrame, criteria: str, attribute: int):
    from pymerkle import verify_inclusion, InvalidProof
    
    matching_users = []

    query_tree = FixmeTree(algorithm='sha256')
    query_tree.append_entry(criteria)
    base = query_tree.get_leaf(1)

    for i in range(len(dataframe)):
        user_tree = dataframe.iloc[i][0]

        proof = user_tree.prove_inclusion(attribute, 8)
        root = user_tree.get_state(8)

        try:
            verify_inclusion(base, root, proof)
            matching_users.append(i)
        except InvalidProof:
            continue

    return matching_users

In [ ]:
london_users_lists = test_geolocation_match(hex_data, 'UK-London', 8)
london_users = df.iloc[london_users_lists]
london_users

,id,user_full_name,city,gender,age,email,phone,bookingtime,complaint,satisfaction_rating,allergy
0,1,User 1,London,male,66,user1@example.com,35840958516,2025-01-01 09:00:00,Burning sensation,4,Pollen
11,12,User 12,London,other,46,user12@example.com,35840332148,2025-01-02 18:00:00,Headache,4,Gluten
17,18,User 18,London,other,77,user18@example.com,35840913619,2025-01-03 12:00:00,Flu-like symptoms,1,Pollen
24,25,User 25,London,female,65,user25@example.com,35840155391,2025-01-04 09:00:00,Skin rash,3,Pollen
25,26,User 26,London,other,47,user26@example.com,35840589236,2025-01-04 12:00:00,Flu-like symptoms,4,Penicillin
...,...,...,...,...,...,...,...,...,...,...,...
9984,9985,User 9985,London,female,82,user9985@example.com,35840700983,2028-06-02 09:00:00,Headache,2,Pollen
9990,9991,User 9991,London,female,53,user9991@example.com,35840839144,2028-06-03 03:00:00,Flu-like symptoms,1,Penicillin
9993,9994,User 9994,London,female,30,user9994@example.com,35840802161,2028-06-03 12:00:00,Burning sensation,5,Gluten
9994,9995,User 9995,London,other,34,user9995@example.com,35840987217,2028-06-03 15:00:00,Flu-like symptoms,1,Pollen
